In [0]:
%run ./config

In [0]:
df_schema= StructType([
    StructField('date', DateType(), True),
    StructField('sku_name', StringType(), True),
    StructField('sku_id', StringType(), True),
    StructField('total_units', DoubleType(), True),
    StructField('price', DoubleType(), True),
    StructField('total_revenue', DoubleType(), True),
    StructField('data_source', StringType(), True)
])

In [0]:
myntra_df= spark.read.schema(df_schema).option('header', True).csv(silver+'/myntra.csv')

In [0]:
myntra_df=myntra_df.groupBy('date', 'sku_name', 'sku_id')\
    .agg(sum('total_units').alias('total_units'), \
     sum('total_revenue').alias('total_revenue'),\
     lit('myntra').alias('data_source'))\
    .orderBy(asc("date"))

In [0]:
blinkit_df=spark.read.schema(df_schema).option('header', True).csv(silver+'/blinkit.csv')

In [0]:
blinkit_df=blinkit_df.groupBy('date', 'sku_id', 'sku_name')\
                    .agg(sum('total_units').alias('total_units'), \
                     sum('total_revenue').alias('total_revenue'),\
                     lit('blinkit').alias('data_source'))\
                    .orderBy(asc("date"))

In [0]:
zepto_df=spark.read.schema(df_schema).option('header', True).csv(silver+'/zepto.csv')

In [0]:
zepto_df=zepto_df.groupBy('date', 'sku_name', 'sku_id')\
                .agg(sum('total_units').alias('total_units'), \
                 sum('total_revenue').alias('total_revenue'),\
                 lit('zepto').alias('data_source'))\
                .orderBy(asc("date"))

In [0]:
nyka_df=spark.read.schema(df_schema).option('header', True).csv(silver+'/nyka.csv')

In [0]:
nyka_df=nyka_df.groupBy('date', 'sku_name', 'sku_id')\
                .agg(sum('total_units').alias('total_units'), \
                 round(sum('total_revenue'),2).alias('total_revenue'),\
                 lit('nyka').alias('data_source'))\
                .orderBy(asc("date"))

In [0]:
df1=blinkit_df.unionByName(myntra_df).unionByName(nyka_df).unionByName(zepto_df)

In [0]:
df1=df1.drop('sku_name').withColumn('product_identifier', concat_ws('_', 'sku_id', 'date'))

In [0]:
df1=df1.select("date", "product_identifier", round(col("total_units"),2).alias("total_units"), "total_revenue", "data_source" )